# 03 — Predict no-shows and choose the model honestly

Part 3 (30 marks). Spec: `problem-statment.md` → Part 3. **The test set is for reporting only** — never used to choose a model, setting or threshold. Rows are not shuffled or re-sorted after loading.

## Setup — imports & paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

root = Path.cwd().parent                     # notebook lives in "date processing/"
clean_csv = root / "appointments_clean.csv"
scores_csv = root / "test_risk_scores.csv"

## Task 3.1 — Features, split and baseline

### 3.1.1 Load `appointments_clean.csv`

In [2]:
df = pd.read_csv(clean_csv)                  # no shuffle / sort after this: test counts depend on row order
print(df.shape)
df.head()

(990, 16)


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


### 3.1.2 Build `X` & `y`

In [3]:
features = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
            "reminder_sent", "booking_channel", "visit_type"]
X = df[features].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

print(f"X has {X.shape[1]} columns:", list(X.columns))

X has 11 columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


### 3.1.3 Train/test split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 3.1.4 Sizes, test no-shows, baseline accuracy

In [5]:
baseline_acc = (y_test == 0).mean()          # always predict "Attended" (0)
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")
print(f"No-shows in test: {y_test.sum()} | overall no-show rate: {100 * y.mean():.1f}%")
print(f"Baseline accuracy (always 'Attended'): {baseline_acc:.4f} ({(y_test == 0).sum()} of {len(y_test)})")

Train rows: 792 | Test rows: 198
No-shows in test: 49 | overall no-show rate: 22.0%
Baseline accuracy (always 'Attended'): 0.7525 (149 of 198)


## Task 3.2 — Logistic regression

### 3.2.1 Scale (fit on train only)

In [6]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)    # mean/SD learned from train only
X_test_s = scaler.transform(X_test)          # test reuses the train mean/SD

### 3.2.2 Fit & test metrics

In [7]:
def report(name, y_true, y_pred, y_prob):
    """Print the classification metrics used throughout Part 3 and return them as a dict."""
    m = {"accuracy": accuracy_score(y_true, y_pred),
         "precision": precision_score(y_true, y_pred, zero_division=0),
         "recall": recall_score(y_true, y_pred),
         "f1": f1_score(y_true, y_pred),
         "roc_auc": roc_auc_score(y_true, y_prob)}
    print(name, {k: round(v, 3) for k, v in m.items()})
    return m

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
lr_pred = model.predict(X_test_s)
lr_prob = model.predict_proba(X_test_s)[:, 1]

lr_metrics = report("Logistic regression (test):", y_test, lr_pred, lr_prob)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, lr_pred))

Logistic regression (test): {'accuracy': 0.788, 'precision': 0.63, 'recall': 0.347, 'f1': 0.447, 'roc_auc': 0.789}
Confusion matrix [[TN, FP], [FN, TP]]:
[[139  10]
 [ 32  17]]


### 3.2.3 Coefficients & `reminder_sent` odds (scaled vs per No→Yes)

In [8]:
coefs = pd.Series(model.coef_[0], index=X.columns)
print(coefs.round(3).to_string())

i = list(X.columns).index("reminder_sent")
coef_r = model.coef_[0][i]
sd_r = scaler.scale_[i]
print(f"\nreminder_sent coefficient (scaled): {coef_r:.3f}  | SD of reminder_sent: {sd_r:.3f}")
print(f"np.exp(coef)          = {np.exp(coef_r):.2f}  -> odds x this per 1 SD of the scaled feature")
print(f"np.exp(coef / SD)     = {np.exp(coef_r / sd_r):.2f}  -> odds x this for No (0) -> Yes (1)")
print(f"i.e. about {100 * (1 - np.exp(coef_r / sd_r)):.0f}% lower odds of a no-show with a reminder")

age                       -0.254
distance_km                0.253
fee_inr                    0.453
lead_days                  0.614
prior_no_shows             0.650
reminder_sent             -0.581
booking_channel_App       -0.047
booking_channel_Phone      0.216
booking_channel_Walk-In   -0.186
visit_type_Consultation    0.016
visit_type_Follow-up      -0.016

reminder_sent coefficient (scaled): -0.581  | SD of reminder_sent: 0.495
np.exp(coef)          = 0.56  -> odds x this per 1 SD of the scaled feature
np.exp(coef / SD)     = 0.31  -> odds x this for No (0) -> Yes (1)
i.e. about 69% lower odds of a no-show with a reminder


**Answer:** `reminder_sent` has the most negative coefficient (−0.581), and `lead_days` (+0.614) and `prior_no_shows` (+0.650) are clearly positive. `np.exp(−0.581)` = **0.56** is the odds ratio for a one-**standard-deviation** rise in the scaled feature (SD = 0.495). Dividing by that SD first, `np.exp(−0.581 / 0.495)` = **0.31**, is the odds ratio for the real change from No (0) to Yes (1). The two differ because the model was fitted on standardised features, so its coefficient measures a change of about half a unit, not the full 0→1 step. **Conclusion:** in this model, holding the other features constant, appointments with a reminder have about **69% lower odds** of a no-show. This is an association in observational data, not proof that reminders cause fewer no-shows.

## Task 3.3 — Decision tree and overfitting

### 3.3.1 Unlimited-depth tree

In [9]:
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print(f"Full tree (depth {tree_full.get_depth()}): "
      f"train acc {tree_full.score(X_train, y_train):.3f} | test acc {tree_full.score(X_test, y_test):.3f} "
      f"| baseline {baseline_acc:.4f}")

Full tree (depth 15): train acc 1.000 | test acc 0.672 | baseline 0.7525


### 3.3.2 `max_depth=3` tree

In [10]:
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(f"Depth-3 tree: train acc {tree3.score(X_train, y_train):.3f} | test acc {tree3.score(X_test, y_test):.3f}")

Depth-3 tree: train acc 0.822 | test acc 0.778


**Answer:** The unlimited tree grows to depth 15 and scores **1.000 on training but 0.672 on test**, which is *below* the 0.7525 always-'Attended' baseline: the large gap shows it memorised noise in the training rows instead of learning patterns that generalise. Limiting it to depth 3 gives **0.822 train / 0.778 test**, so the gap is small and the test accuracy beats the baseline.

## Task 3.4 — Random forest and a comparison table

### 3.4.1 Random forest metrics

In [11]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print(f"Random forest train acc: {rf.score(X_train, y_train):.3f}")
rf_metrics = report("Random forest (test):", y_test, rf.predict(X_test), rf.predict_proba(X_test)[:, 1])

Random forest train acc: 1.000
Random forest (test): {'accuracy': 0.747, 'precision': 0.484, 'recall': 0.306, 'f1': 0.375, 'roc_auc': 0.706}


### 3.4.2 Top-3 feature importances

In [12]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(3).round(3)

lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64

### 3.4.3 Comparison table

In [13]:
tree3_metrics = report("Depth-3 tree (test):", y_test, tree3.predict(X_test), tree3.predict_proba(X_test)[:, 1])

comparison = pd.DataFrame({
    "Logistic regression": lr_metrics,
    "Decision tree (depth 3)": tree3_metrics,
    "Random forest": rf_metrics,
}).T[["accuracy", "recall", "roc_auc"]].round(3)
comparison

Depth-3 tree (test): {'accuracy': 0.778, 'precision': 0.647, 'recall': 0.224, 'f1': 0.333, 'roc_auc': 0.7}


,accuracy,recall,roc_auc
Logistic regression,0.788,0.347,0.789
Decision tree (depth 3),0.778,0.224,0.700
Random forest,0.747,0.306,0.706


**Answer:** Logistic regression looks best on every column: highest accuracy (0.788), recall (0.347) and ROC-AUC (0.789). The random forest fits the training data perfectly (1.000) but drops to 0.747 test accuracy and 0.706 ROC-AUC, so it overfits. Its top three importances are `lead_days` (0.220), `age` (0.197) and `distance_km` (0.195). This table is for reporting only, because it uses the test set; the model is formally chosen in 3.5 from training data alone.

## Task 3.5 — Tune, choose the model and threshold, then test once

### 3.5(a) GridSearchCV on train

In [14]:
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)                   # training rows only
print("best_params_:", grid.best_params_)
print(f"best_score_ (CV ROC-AUC): {grid.best_score_:.3f}")

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (CV ROC-AUC): 0.767


### 3.5(b) Cross-validated model choice

In [15]:
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

candidates = {
    "Logistic regression": lr_pipe,          # unscaled X_train: scaler re-fitted inside each fold
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=42),
}
cv_auc = {name: cross_val_score(m, X_train, y_train, cv=5, scoring="roc_auc").mean()
          for name, m in candidates.items()}
cv_auc["Tuned random forest"] = grid.best_score_

cv_table = pd.Series(cv_auc, name="cv_roc_auc").sort_values(ascending=False).round(3)
winner_name = cv_table.index[0]
print(cv_table.to_string())
print("\nWinner (highest CV ROC-AUC on train):", winner_name)

Logistic regression        0.774
Tuned random forest        0.767
Random forest              0.747
Decision tree (depth 3)    0.678

Winner (highest CV ROC-AUC on train): Logistic regression


### 3.5(c) Threshold choice from out-of-fold probabilities

In [16]:
winner = lr_pipe if winner_name == "Logistic regression" else grid.best_estimator_
oof = cross_val_predict(winner, X_train, y_train, cv=5, method="predict_proba")[:, 1]

rows = []
for t in (0.5, 0.35):
    flag = (oof >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_train, flag),
                 "recall": recall_score(y_train, flag),
                 "f1": f1_score(y_train, flag),
                 "flagged": flag.sum()})
thresholds = pd.DataFrame(rows).set_index("threshold").round(3)
threshold = thresholds["f1"].idxmax()        # higher F1 wins
print(thresholds)
print("\nChosen threshold:", threshold)

           precision  recall     f1  flagged
threshold                                   
0.50           0.575   0.249  0.347       73
0.35           0.513   0.467  0.489      154

Chosen threshold: 0.35


**Answer:** Logistic regression has the highest cross-validated ROC-AUC on the training set (0.774, against 0.767 for the tuned forest), so it is the winner. On out-of-fold training probabilities, **0.35 is chosen**: its F1 is 0.489 against 0.347 at 0.5, and its recall nearly doubles (0.467 vs 0.249), at the cost of flagging 154 bookings instead of 73. Recall matters more here because a missed warning costs an empty clinic slot, while an extra call costs the front desk about a minute.

### 3.5(d) Fit winner, evaluate ONCE on test

In [17]:
winner.fit(X_train, y_train)                 # pipeline fits its own scaler on train
probs = winner.predict_proba(X_test)[:, 1]
flags = (probs >= threshold).astype(int)

print(f"Test set, {winner_name} @ {threshold}:")
print(f"precision {precision_score(y_test, flags):.3f} | recall {recall_score(y_test, flags):.3f} "
      f"| flagged {flags.sum()} of {len(flags)} | ROC-AUC {roc_auc_score(y_test, probs):.3f}")

Test set, Logistic regression @ 0.35:
precision 0.571 | recall 0.571 | flagged 49 of 198 | ROC-AUC 0.789


### 3.5(e) Save `test_risk_scores.csv`

In [18]:
risk = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs.round(4),
    "flag": flags,
})
risk.to_csv(scores_csv, index=False)
print(f"Saved {scores_csv.name}: {len(risk)} rows, {risk['flag'].sum()} flagged")
risk.sort_values("risk_prob", ascending=False).head()

Saved test_risk_scores.csv: 198 rows, 49 flagged


,appointment_id,no_show,risk_prob,flag
68,A0635,1,0.9372,1
24,A0544,1,0.9241,1
141,A0293,1,0.9196,1
106,A0950,1,0.8186,1
114,A0996,0,0.7667,1


**Answer:** On the single test-set evaluation at 0.35, logistic regression flags **49 of 198** appointments with precision 0.571, recall 0.571 and ROC-AUC 0.789; `test_risk_scores.csv` has 198 rows and 49 with `flag = 1`. A real clinic would choose the threshold from how many follow-up calls the front desk can make in an evening, weighed against the cost of an empty slot, rather than from F1 alone.